<a href="https://colab.research.google.com/github/faith-dev122/BIT4133-NLP-Project/blob/main/Creative_Consolidation_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# CREATIVE CONSOLIDATION PROJECT
# Combining Week 6 (Word2Vec) + Week 7 (Neural Language Model)
# Setup: Install Gensim
!pip install gensim --quiet

import gensim
print("Gensim version:", gensim.__version__)
print("Ready for Word2Vec!")

Gensim version: 4.4.0
Ready for Word2Vec!


In [5]:
# NEW CONCEPT: Transfer Learning using Word2Vec embeddings
# as pre-trained weights inside a neural network
import numpy as np
from gensim.models import Word2Vec
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

print("=" * 65)
print("  CREATIVE PROJECT: Word2Vec + Neural Language Model")
print("  New Concept: Transfer Learning with Pre-trained Embeddings")
print("=" * 65)

# ---- STEP 1: Prepare text data ----
text_data = """
natural language processing helps computers understand human language
machine learning is a part of artificial intelligence
deep learning uses neural networks to process data
nlp is used in chatbots and search engines
word embeddings capture the meaning of words
neural networks learn patterns from large amounts of data
the student is studying for the exam
the cat drinks milk every morning
the dog plays in the garden every day
artificial intelligence is transforming the world
"""

# ---- STEP 2: Tokenize with Keras ----
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text_data])
total_words = len(tokenizer.word_index) + 1
print(f"\nVocabulary size: {total_words}")

# ---- STEP 3: Train Word2Vec on the SAME text (Week 6 concept) ----
sentences_for_w2v = [line.split() for line in text_data.split('\n') if line.strip()]
w2v_model = Word2Vec(sentences_for_w2v, vector_size=16,
                      window=2, min_count=1, workers=4)

print(f"Word2Vec trained on {len(sentences_for_w2v)} sentences")
print(f"Embedding dimension: 16")

# ---- STEP 4: Build embedding matrix from Word2Vec ----
# This is the NEW CONCEPT — transferring Word2Vec weights
# into the neural network's Embedding layer
embedding_dim = 16
embedding_matrix = np.zeros((total_words, embedding_dim))

for word, idx in tokenizer.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]

print(f"\nEmbedding matrix shape: {embedding_matrix.shape}")
print("This matrix contains Word2Vec's learned meaning for each word —")
print("instead of starting the neural network with random weights,")
print("we START it with weights that already understand word meaning!")

# ---- STEP 5: Prepare training sequences (Week 7 concept) ----
input_sequences = []
for line in text_data.split('\n'):
    if line.strip() == "":
        continue
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

max_seq_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

X = input_sequences[:, :-1]
y = to_categorical(input_sequences[:, -1], num_classes=total_words)

# ---- STEP 6: Build neural network WITH Word2Vec embeddings ----
model_combined = Sequential([
    Embedding(total_words, embedding_dim,
              weights=[embedding_matrix],   # <-- Word2Vec weights loaded here!
              input_length=max_seq_len-1,
              trainable=True),               # allow fine-tuning
    LSTM(32),
    Dense(total_words, activation='softmax')
])

model_combined.compile(loss='categorical_crossentropy',
                        optimizer='adam', metrics=['accuracy'])

model_combined.summary()
print("\nModel built using Word2Vec embeddings as initial weights!")

  CREATIVE PROJECT: Word2Vec + Neural Language Model
  New Concept: Transfer Learning with Pre-trained Embeddings

Vocabulary size: 56
Word2Vec trained on 10 sentences
Embedding dimension: 16

Embedding matrix shape: (56, 16)
This matrix contains Word2Vec's learned meaning for each word —
instead of starting the neural network with random weights,
we START it with weights that already understand word meaning!


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 896 (3.50 KB)

 Trainable params: 896 (3.50 KB)

 Non-trainable params: 0 (0.00 B)


Model built using Word2Vec embeddings as initial weights!


In [6]:
# Train and test
print("TRAINING THE COMBINED MODEL...")
history = model_combined.fit(X, y, epochs=100, verbose=0)
print(f"Final training accuracy: {history.history['accuracy'][-1]*100:.2f}%")

# Test predictions
def predict_with_w2v_model(seed_text, top_n=3):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')
    probs = model_combined.predict(token_list, verbose=0)[0]
    top_indices = probs.argsort()[-top_n:][::-1]

    print(f"\nInput: '{seed_text}'")
    for idx in top_indices:
        word = tokenizer.index_word.get(idx, '?')
        print(f"  → '{word}' ({probs[idx]*100:.1f}%)")

print("\n" + "=" * 55)
print("PREDICTIONS USING WORD2VEC-INITIALISED MODEL")
print("=" * 55)
for phrase in ["the cat drinks", "machine learning is", "natural language"]:
    predict_with_w2v_model(phrase)

print("\n\nKEY TAKEAWAY:")
print("-" * 55)
print("Week 6 taught us Word2Vec creates meaningful word vectors.")
print("Week 7 taught us neural networks predict sequences.")
print("This project shows these are NOT separate techniques —")
print("Word2Vec embeddings can be FED INTO a neural network as")
print("a starting point. This is called TRANSFER LEARNING and")
print("it is exactly how BERT, GPT and modern LLMs are built —")
print("they start with pre-trained embeddings, then fine-tune")
print("on specific tasks. This project demonstrates that bridge")
print("between Week 6 and Week 7 concepts.")

TRAINING THE COMBINED MODEL...
Final training accuracy: 21.54%

PREDICTIONS USING WORD2VEC-INITIALISED MODEL

Input: 'the cat drinks'
  → 'the' (14.9%)
  → 'is' (9.7%)
  → 'language' (5.4%)

Input: 'machine learning is'
  → 'is' (30.3%)
  → 'the' (10.9%)
  → 'learning' (10.3%)

Input: 'natural language'
  → 'is' (21.8%)
  → 'learning' (10.3%)
  → 'the' (8.6%)


KEY TAKEAWAY:
-------------------------------------------------------
Week 6 taught us Word2Vec creates meaningful word vectors.
Week 7 taught us neural networks predict sequences.
This project shows these are NOT separate techniques —
Word2Vec embeddings can be FED INTO a neural network as
a starting point. This is called TRANSFER LEARNING and
it is exactly how BERT, GPT and modern LLMs are built —
they start with pre-trained embeddings, then fine-tune
on specific tasks. This project demonstrates that bridge
between Week 6 and Week 7 concepts.
